In [0]:
# =====================================================================
# proceso / 03_ingest_calendar.py
# Task "ingest_calendar" (Bronze - Extract). PySpark puro.
# Tercera fuente del proyecto: feriados federales de EE.UU. Se genero
# con la libreria "holidays" (2014-2026) en vez de usar un CSV de
# Kaggle, porque los datasets de feriados encontrados en Kaggle no
# cubrian el rango de fechas real del dataset de e-commerce (2023-2025).
# El archivo generado vive en datasets/calendar/us_holidays_2014_2026.csv
# y se sube tal cual al contenedor raw/calendar/. Corre en paralelo con
# ingest_superstore e ingest_ecommerce (las 3 dependen solo de prepamb)
# y converge en "transform", igual patrón que las otras dos ingestas.
# =====================================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("raw_path", "abfss://raw@adlsretailproject0826.dfs.core.windows.net/calendar/")
dbutils.widgets.text("catalogo", "retail_medallion")
raw_path = dbutils.widgets.get("raw_path")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
# Schema del calendario generado: Date, Holiday
calendar_schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Holiday", StringType(), True),
])

In [0]:
df_raw = (
    spark.read
    .option("header", True)
    .schema(calendar_schema)
    .csv(raw_path)
)

df_bronze = (
    df_raw
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source_system", F.lit("calendar_kaggle"))
    .select(  # mismo orden que bronze.calendar_holidays en PrepAmb
        "Date", "Holiday", "_source_file", "_ingest_timestamp", "_source_system",
    )
)

In [0]:
df_bronze.write.mode("overwrite").insertInto(f"{catalogo}.bronze.calendar_holidays")

print(f"Bronze OK -> {catalogo}.bronze.calendar_holidays ({df_bronze.count()} filas)")

Bronze OK -> retail_medallion.bronze.calendar_holidays (151 filas)
